# Fall Risk Detection System - Getting Started

This notebook demonstrates the core functionality of the Fall Risk Detection System
for analyzing home environments and identifying fall hazards for elderly individuals.

## System Overview

The system uses Vision-Language Models (VLMs) to:
1. Analyze images of home environments
2. Detect fall hazards based on Westmead Home Safety Assessment criteria
3. Generate quantified risk scores (0-100)
4. Provide explainable outputs with recommendations

## Prerequisites

Before running this notebook:
1. Install dependencies: `pip install -r requirements.txt`
2. Create a `.env` file with your API key:
   - For OpenAI: `OPENAI_API_KEY=your_key_here`
   - For Google Gemini: `GOOGLE_API_KEY=your_key_here`

## 1. Setup and Imports

In [ ]:
# Add project root to path
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(project_root / '.env')

print(f"Project root: {project_root}")

In [ ]:
# Core imports
import os
import json
from pprint import pprint

# Visualization
import matplotlib.pyplot as plt
from PIL import Image

# Project modules
from src.models.base_model import ImageInput, HazardDetectionResult
from src.models.model_factory import VisionModelFactory, ModelType
from src.hazard_detection.detector import HazardDetector, DetectionConfig
from src.hazard_detection.categories import HAZARD_DEFINITIONS, HazardCategory
from src.risk_scoring.scorer import RiskScorer, RiskLevel

## 2. Initialize the Vision Model

Choose between OpenAI GPT-4 Vision or Google Gemini.

In [ ]:
# Check which API keys are available
openai_available = bool(os.getenv('OPENAI_API_KEY'))
google_available = bool(os.getenv('GOOGLE_API_KEY'))

print(f"OpenAI API available: {openai_available}")
print(f"Google Gemini API available: {google_available}")

In [ ]:
# Initialize the vision model
# Uncomment the model you want to use:

# Option 1: OpenAI GPT-4 Vision (recommended for best accuracy)
# model = VisionModelFactory.create(ModelType.OPENAI, {
#     'model_id': 'gpt-4o',
#     'detail': 'high'
# })

# Option 2: Google Gemini (good alternative, potentially lower cost)
# model = VisionModelFactory.create(ModelType.GEMINI, {
#     'model_id': 'gemini-2.0-flash'
# })

# Option 3: Auto-detect based on available API keys
model = VisionModelFactory.create_auto({})

print(f"Using model: {model.model_name}")

In [ ]:
# Verify model connectivity
health_ok = model.health_check()
print(f"Model health check: {'PASSED' if health_ok else 'FAILED'}")

## 3. Initialize Hazard Detector and Risk Scorer

In [ ]:
# Initialize the hazard detector
detector = HazardDetector(
    vision_model=model,
    prompts_dir=str(project_root / 'configs' / 'prompts')
)

# Initialize the risk scorer
scorer = RiskScorer()

print("Detector and scorer initialized successfully!")

## 4. Load and Display a Test Image

Place your test images in the `data/sample/` directory, or specify a path to any image.

In [ ]:
# List available sample images
sample_dir = project_root / 'data' / 'sample'
sample_dir.mkdir(parents=True, exist_ok=True)

sample_images = list(sample_dir.glob('*.jpg')) + list(sample_dir.glob('*.png'))
print(f"Found {len(sample_images)} sample images:")
for img in sample_images:
    print(f"  - {img.name}")

if not sample_images:
    print("\nNo sample images found. Please add images to data/sample/")
    print("You can use any home environment image (bathroom, kitchen, living room, etc.)")

In [ ]:
# Set the image path - modify this to use your image
# Option 1: Use a sample image
# image_path = sample_dir / 'your_image.jpg'

# Option 2: Use any image path
image_path = "path/to/your/image.jpg"  # <-- MODIFY THIS

# Verify the image exists
if Path(image_path).exists():
    print(f"Using image: {image_path}")
else:
    print(f"Image not found: {image_path}")
    print("Please set a valid image path above.")

In [ ]:
# Display the image
if Path(image_path).exists():
    img = Image.open(image_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Test Image for Hazard Analysis')
    plt.show()
    
    print(f"Image size: {img.size}")
    print(f"Image format: {img.format}")

## 5. Run Hazard Detection

Analyze the image to detect fall hazards.

In [ ]:
# Create image input
image_input = ImageInput(path=str(image_path))

# Configure detection
config = DetectionConfig(
    use_chain_of_thought=True,  # Better reasoning
    multi_pass=True,            # Multiple analysis passes
    confidence_threshold=0.5,   # Minimum confidence
    temperature=0.1,            # Low temperature for consistency
)

print("Running hazard detection...")
print("(This may take 10-30 seconds depending on the model)")

In [ ]:
# Run detection
detection_result = detector.detect_hazards(image_input, config)

print(f"\nDetection complete!")
print(f"Processing time: {detection_result.processing_time_ms:.0f}ms")
print(f"Room type identified: {detection_result.room_type}")
print(f"Total hazards detected: {len(detection_result.hazards)}")
print(f"Overall confidence: {detection_result.overall_confidence:.2f}")

## 6. View Detected Hazards

In [ ]:
# Display all detected hazards
print("\n" + "="*60)
print("DETECTED HAZARDS")
print("="*60)

for i, hazard in enumerate(detection_result.hazards, 1):
    print(f"\n[{i}] {hazard.category.upper()} - {hazard.subcategory}")
    print(f"    Severity: {hazard.severity.upper()}")
    print(f"    Confidence: {hazard.confidence:.2f}")
    print(f"    Description: {hazard.description}")
    print(f"    Location: {hazard.location_description}")
    if hazard.recommendations:
        print(f"    Recommendations:")
        for rec in hazard.recommendations:
            print(f"      - {rec}")

In [ ]:
# Group hazards by severity
hazards_by_severity = detection_result.hazards_by_severity

print("\nHazards by Severity:")
for severity in ['critical', 'high', 'medium', 'low']:
    hazards = hazards_by_severity.get(severity, [])
    if hazards:
        print(f"  {severity.upper()}: {len(hazards)}")

## 7. Calculate Risk Score

In [ ]:
# Calculate the risk score
risk_result = scorer.calculate_score(detection_result)

print("\n" + "="*60)
print("RISK ASSESSMENT SUMMARY")
print("="*60)
print(f"\nOVERALL RISK SCORE: {risk_result.total_score}/100")
print(f"RISK LEVEL: {risk_result.risk_level.value.upper()}")

In [ ]:
# Display risk level with color coding
risk_colors = {
    RiskLevel.LOW: 'green',
    RiskLevel.MODERATE: 'orange',
    RiskLevel.HIGH: 'darkorange',
    RiskLevel.CRITICAL: 'red'
}

# Create a visual risk meter
fig, ax = plt.subplots(figsize=(10, 2))

# Background gradient
gradient = ax.imshow([[0, 0.33, 0.66, 1]], cmap='RdYlGn_r', aspect='auto',
                     extent=[0, 100, 0, 1])

# Score marker
ax.axvline(x=risk_result.total_score, color='black', linewidth=3)
ax.plot(risk_result.total_score, 0.5, 'ko', markersize=15)

# Labels
ax.set_xlim(0, 100)
ax.set_ylim(0, 1)
ax.set_xlabel('Risk Score', fontsize=12)
ax.set_yticks([])
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xticklabels(['0\nLow', '25', '50\nModerate', '75', '100\nCritical'])
ax.set_title(f'Fall Risk Score: {risk_result.total_score}/100 ({risk_result.risk_level.value.upper()})',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Category breakdown
print("\nRisk Score by Category:")
print("-" * 40)

for cat_name, cat_score in sorted(
    risk_result.category_breakdown.items(),
    key=lambda x: x[1].weighted_score,
    reverse=True
):
    if cat_score.hazard_count > 0:
        print(f"{cat_name.upper():12} | Score: {cat_score.weighted_score:5.1f} | "
              f"Hazards: {cat_score.hazard_count} | Max Severity: {cat_score.max_severity}")

## 8. View Recommendations

In [ ]:
# Display prioritized recommendations
print("\n" + "="*60)
print("PRIORITIZED RECOMMENDATIONS")
print("="*60)

for i, rec in enumerate(risk_result.recommendations_summary, 1):
    print(f"{i}. {rec}")

In [ ]:
# Display full explanation
print("\n" + "="*60)
print("DETAILED EXPLANATION")
print("="*60)
print(risk_result.explanation)

## 9. Export Results

Save the detection and scoring results for documentation.

In [ ]:
# Prepare results for export
results_export = {
    "image_path": str(image_path),
    "model_used": detection_result.model_name,
    "processing_time_ms": detection_result.processing_time_ms,
    "room_type": detection_result.room_type,
    "detection_confidence": detection_result.overall_confidence,
    "risk_score": risk_result.total_score,
    "risk_level": risk_result.risk_level.value,
    "total_hazards": len(detection_result.hazards),
    "hazards": [
        {
            "category": h.category,
            "subcategory": h.subcategory,
            "description": h.description,
            "severity": h.severity,
            "confidence": h.confidence,
            "location": h.location_description,
            "recommendations": h.recommendations
        }
        for h in detection_result.hazards
    ],
    "recommendations_summary": risk_result.recommendations_summary,
    "explanation": risk_result.explanation
}

# Save to JSON
output_path = project_root / 'reports' / 'results' / f'{Path(image_path).stem}_analysis.json'
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(results_export, f, indent=2)

print(f"Results saved to: {output_path}")

## 10. Analyze Multiple Images (Optional)

Batch analyze multiple images in a directory.

In [ ]:
# Function to analyze multiple images
def analyze_batch(image_dir, detector, scorer, max_images=10):
    """
    Analyze multiple images and return summary statistics.
    """
    image_dir = Path(image_dir)
    images = list(image_dir.glob('*.jpg')) + list(image_dir.glob('*.png'))
    images = images[:max_images]
    
    results = []
    
    for img_path in images:
        print(f"Analyzing: {img_path.name}")
        try:
            image_input = ImageInput(path=str(img_path))
            detection = detector.detect_quick(image_input)
            score = scorer.calculate_score(detection)
            
            results.append({
                'image': img_path.name,
                'room_type': detection.room_type,
                'hazards': len(detection.hazards),
                'risk_score': score.total_score,
                'risk_level': score.risk_level.value
            })
            print(f"  -> Score: {score.total_score}, Level: {score.risk_level.value}")
        except Exception as e:
            print(f"  -> Error: {e}")
    
    return results

# Uncomment to run batch analysis:
# batch_results = analyze_batch(sample_dir, detector, scorer)

## Summary

This notebook demonstrated:
1. Setting up the Fall Risk Detection System
2. Analyzing home environment images
3. Detecting fall hazards using vision-language models
4. Calculating quantified risk scores
5. Generating actionable recommendations

### Next Steps
- Add more test images to `data/sample/`
- Experiment with different detection configurations
- Compare results between different vision models
- Create annotated ground truth for evaluation